# Calling an LLM: Without a Framework vs With LangChain

This notebook compares two ways to call Groq-hosted `openai/gpt-oss-120b`:

1. **Without a framework:** use the Groq Python SDK directly.
2. **With LangChain:** use `init_chat_model()` and `model.invoke()`.


## Key idea

Both examples call a chat model. The main difference is the abstraction layer: the Groq SDK exposes provider-specific request/response objects, while LangChain gives a more uniform model interface that can be reused with other integrations and LangChain components.


## 1. Install dependencies

Run this cell once in the notebook environment.

In [ ]:
%pip install -U groq langchain langchain-groq python-dotenv

## 2. Configure the API key

`load_dotenv()` loads values from a `.env` file in the working directory (or a parent directory, depending on dotenv behavior). Keep your key private; don't paste it directly into notebook cells or commit `.env` to Git.

Your `.env` file should contain:

```text
GROQ_API_KEY=your_actual_key_here
```


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY not found. Add it to your .env file.")


## 3. Without a framework — Groq SDK

### Concepts

- `Groq(api_key=...)`: creates a provider-specific API client.
- `chat.completions.create(...)`: sends a chat completion request.
- `messages`: conversation messages with roles such as `user` and `assistant`.
- `temperature`: controls response variability.
- `max_completion_tokens`: caps generated completion tokens (model/provider support can vary).
- `stream=True`: returns chunks as they arrive, instead of waiting for the whole response.
- `chunk.choices[0].delta.content`: text in the current streamed chunk.

`for chunk in client` loops over the incoming stream. `end=""` avoids adding a newline after every chunk; `flush=True` displays output promptly.


In [ ]:
from groq import Groq

groq_client = Groq(api_key=api_key)

def chat_bot(user_input: str) -> None:
    stream = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": user_input}],
        temperature=1,
        max_completion_tokens=2048,
        stream=True,
    )

    for chunk in stream:
        text = chunk.choices[0].delta.content or ""
        print(text, end="", flush=True)
    print()  # finish output on a new line

user_input = input("Enter a query: ")
chat_bot(user_input)


### What happens in the SDK version?

1. User enters a prompt.
2. Your code constructs a provider-specific request.
3. Groq returns a stream of chunks.
4. Your loop extracts and prints each chunk.

The code explicitly handles streaming, so the response appears progressively.


## 4. With LangChain — model abstraction

### Concepts

- `init_chat_model(...)`: initializes a chat model through LangChain's integration layer.
- `model_provider="groq"`: selects the Groq integration.
- `model.invoke(...)`: sends a prompt and returns a message object.
- `response.content`: reads the generated text.

LangChain makes model calling more consistent across providers and fits naturally with prompts, output parsers, tools, retrievers, and agents. In this simple example, `invoke()` waits for the response; use `model.stream(...)` when you want streaming.


In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    api_key=api_key,
    temperature=0.7,
    max_tokens=1000,
)

user_input = input("Enter a query: ")
response = model.invoke(user_input)
print(response.content)


## 5. Side-by-side comparison

| Topic | Direct Groq SDK | LangChain |
|---|---|---|
| Client setup | `Groq(api_key=...)` | `init_chat_model(...)` |
| Request method | `chat.completions.create(...)` | `model.invoke(...)` |
| Response | Provider-specific completion/chunk objects | LangChain message object |
| Streaming | `stream=True`, iterate response chunks | `model.stream(...)` |
| Abstraction | Groq-specific | Common interface across supported integrations |
| Extra ecosystem | Use provider SDK features directly | Integrates with prompts, parsers, tools, retrievers, agents |
| Best fit | Direct API control and provider-specific features | Building applications from reusable LangChain components |


